In [1]:
import pandas as pd
import re

In [2]:
movies = pd.read_csv('../../Data/raw/movies.csv')
ratings = pd.read_csv('../../Data/raw/ratings.csv')

In [3]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
3878,3948,Meet the Parents (2000),Comedy
3879,3949,Requiem for a Dream (2000),Drama
3880,3950,Tigerland (2000),Drama
3881,3951,Two Family House (2000),Drama


In [4]:
ratings

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,5,956704887
1000206,6040,562,5,956704746
1000207,6040,1096,4,956715648


In [5]:
display(ratings['rating'].describe().min())
display(ratings['rating'].describe().max())
display(ratings['rating'].describe().mean())

np.float64(1.0)

np.float64(1000209.0)

np.float64(125028.8373332873)

In [6]:
movies['genres'].value_counts()

genres
Drama                              843
Comedy                             521
Horror                             178
Comedy|Drama                       162
Comedy|Romance                     142
                                  ... 
Drama|Film-Noir                      1
Comedy|Horror|Sci-Fi                 1
Adventure|Drama|Romance|Sci-Fi       1
Adventure|Animation|Sci-Fi           1
Adventure|Crime|Sci-Fi|Thriller      1
Name: count, Length: 301, dtype: int64

In [7]:
len(movies)

3883

In [8]:
id_no_genre = movies[movies['genres'] == '(no genres listed)']['movieId']
movies = movies[movies['genres'] != '(no genres listed)']
ratings = ratings[~ratings['movieId'].isin(id_no_genre)]

In [9]:
def extraer_año(title):
    # Buscar todos los grupos entre paréntesis
    matches = re.findall(r'\((\d{4})\)', title)
    if matches:
        return int(matches[-1])  # Tomar el último grupo encontrado
    return None

In [10]:
movies['year'] = movies['title'].apply(extraer_año)
df_years = movies[['movieId', 'year']].drop_duplicates()

movies["genres"] = movies["genres"].str.split("|")

df_exploded = movies.explode('genres')
genre_dummies = pd.get_dummies(df_exploded['genres'])

df_combined = pd.concat([df_exploded[['movieId', 'title']], genre_dummies], axis=1)

# Paso 3: Agrupar por movieId y title, sumando los valores binarios
df_final_movies = df_combined.groupby(['movieId', 'title'], as_index=False).sum()
df_final_movies = df_final_movies.merge(df_years, on='movieId', how='left')

In [11]:
null_year = df_final_movies[df_final_movies['year'].isna()]['movieId']
ids_ratings = ratings[ratings['movieId'].isin(null_year)].groupby('movieId')['rating'].count().sort_values(ascending=False)

In [12]:
movie_ids_menos_10 = ids_ratings[ids_ratings <= 10]
ids_eliminar = movie_ids_menos_10.index
ids_eliminar

df_final_movies = df_final_movies[~df_final_movies['movieId'].isin(ids_eliminar)]
ratings = ratings[~ratings['movieId'].isin(ids_eliminar)]

In [13]:
movie_ids_mas_10 = ids_ratings[ids_ratings > 10]

ids_peliculas = movie_ids_mas_10.index
display(df_final_movies[df_final_movies['movieId'].isin(ids_peliculas)])

movie_year_dict = {
    133276: 2014,
    140956: 2018,
    147188: 1976,
    149334: 2016,
    150657: 2016,
    162414: 2016,
    165699: 2019,
    168278: 2016,
    169656: 2017,
    174057: 2017,
    183837: 2018,
    183855: 2018,
    187577: 2018,
    188175: 2018,
    189577: 2018,
    190017: 2018,
    190747: 2018,
    197651: 2019,
    198111: 2019,
    198145: 2019,
    203922: 2010,
    204692: 2019,
    205074: 2019
}

,movieId,title,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,...,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year


In [14]:
# Solo actualizamos las filas con year nulo y movieId en el diccionario
df_final_movies.loc[
    df_final_movies['movieId'].isin(movie_year_dict.keys()) & df_final_movies['year'].isna(),
    'year'
] = df_final_movies['movieId'].map(movie_year_dict)

In [16]:
ratings.dropna(inplace=True)
df_final_movies.dropna(inplace=True)

df_final_movies.to_csv('../../Data/transformed/movies_cleaned.csv', index=False)
ratings.to_csv('../../Data/transformed/ratings_cleaned.csv', index=False)